# Agent

The agent loop: one ADK agent whose model (reached through OpenRouter) chooses which
tool to call next and finally writes the review packet. Code does everything else:
every request goes through the gateway, every number comes from a tool field, and the
packet is only accepted once it passes the schema checks and the stray-digit validator.

A run is saved as a JSON record (the requests, the packet, tokens and cost) that can be
replayed offline with no model call: `replay` re-issues each request through a fresh
gateway and checks that every reply's fingerprint matches.

Live runs need `OPENROUTER_API_KEY` (in the shell or a repo-root `.env`); the cells that
call the model are marked `#| notest` so CI never does.

In [ ]:
#| default_exp agent

In [ ]:
#| export
import asyncio
import hashlib
import json
import time
from collections.abc import Mapping
from datetime import UTC, datetime
from pathlib import Path
from typing import Any, Literal

from google.adk.agents import LlmAgent
from google.adk.agents.run_config import RunConfig
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.adk.tools.tool_context import ToolContext
from google.genai import types
from pydantic import BaseModel, ValidationError

from agentic_forecast_review.credentials import (
    openrouter_api_key,
)
from agentic_forecast_review.gateway import (
    Gateway,
)
from agentic_forecast_review.generator import (
    Snapshot,
)
from agentic_forecast_review.schemas import (
    MAX_OBSERVATION_REFS,
    MAX_TURNS,
    Case,
    Claim,
    ClaimLabel,
    Clarification,
    Cost,
    Metric,
    MissingScope,
    Packet,
    RunSetting,
    TerminalState,
    TraceStep,
)
from agentic_forecast_review.tools import (
    TOOLS,
)
from agentic_forecast_review.validator import (
    known_identifiers,
    packet_stray_digits,
)

## Model and prompt

The model is pinned here with its OpenRouter prices (checked on 7 Nov 2026), so every
run records exactly what it used and cost. The prompt's hash goes into each run record;
W5 tags the prompt before the held-out run.

In [ ]:
#| export
MODEL = "openrouter/deepseek/deepseek-v4-flash-0731"
USD_PER_TOKEN = {"prompt": 0.0000000434, "completion": 0.0000003}
TEMPERATURE = 0.0  # matched cases get the same choices until their evidence differs

INSTRUCTION = """\
You review retail forecast performance for an analyst. You see the data only through \
five read-only tools; code computes every number.

How to work:
- Let each result decide your next step. Read what the last tool returned and follow up on what is unusual or unexplained. Do not run a check whose answer a result already gives you, and do not call every tool by habit.
- A worse error only means a worse model if the comparison is fair: the actuals behind it are complete and not duplicated, and both sides use the same model version and the same forecast horizons. compare_runs already reports actual rows found against days expected, the model version and the horizons on each side; check_quality reports missing days, duplicate rows, version labels and horizons; inspect_series shows each day's horizon and model. If the comparison is not fair, stop: do not judge the model.
- Every tool call takes `reason` (one short sentence, no digits) and `based_on`: the one or two earlier call ids whose results made you choose this step, usually the latest one (such as ["cmp2"]); empty for the first call.
- You never write a number or a digit yourself: no counts, days, dates or percentages. Cite tool fields with pointer tokens {ev:<call_id>.<field>}, for example {ev:cmp2.ca_2_foods__mae_period}. Identifiers such as CA_2_FOODS or synthetic_v3 are fine.
- Budget: at most eight turns in all, and each tool call or finish is one turn.

End by calling `finish` once, with one outcome:
- ANSWER: the evidence answers the request.
- CLARIFY: the request leaves the metric, period, window or series unspecified; list \
what is missing. Never ask when the request is fully specified.
- STOP_UNTRUSTWORTHY: the comparison the request relies on is not valid as computed; \
say why.
- INCOMPLETE: you could not establish an answer.
Claims are FINDING (an observed fact, which must cite row IDs returned by the tools) or \
INFERENCE (your interpretation). List under not_established what the tools cannot show, \
such as a cause no tool measures. Never invent a cause.
"""
PROMPT_SHA256 = hashlib.sha256(INSTRUCTION.encode("utf-8")).hexdigest()

## One run

`AgentRun` holds a run's state: the case, its gateway, the trace so far and the
accepted packet. Its methods are what the model can call; each records a trace step
with the model's reason and the earlier calls it cites. References to calls that do
not exist (yet) are dropped and kept in the record.

A request too malformed for the gateway to parse still uses a turn: the step is
recorded without a call, and the refusal is kept in the record.

The model sees each reply without its fingerprint and without the snapshot's ID: both
are identity, not evidence, and the ID alone would make a matched pair's inputs differ.

In [ ]:
#| export
class ClaimInput(BaseModel):
    label: Literal["FINDING", "INFERENCE"]
    text: str
    rows: list[str] = []


def _range(start: int, end: int) -> dict[str, int]:
    return {"start_day": start, "end_day": end}


class AgentRun:
    def __init__(
        self, case: Case, snapshot: Snapshot, setting: RunSetting | None = None
    ):
        self.case = case
        self.snapshot = snapshot
        self.gateway = Gateway(case, snapshot, TOOLS, setting)
        self.steps: list[TraceStep] = []
        self.requests: list[dict[str, Any]] = []
        self.notes: list[str] = []
        self.packet: Packet | None = None

    def _refs(self, based_on: list[str]) -> tuple[str, ...]:
        earlier = {step.call.call_id for step in self.steps if step.call is not None}
        kept = [ref for ref in dict.fromkeys(based_on) if ref in earlier]
        dropped = [ref for ref in based_on if ref not in kept]
        if dropped:
            self.notes.append(f"turn {len(self.steps) + 1}: dropped refs {dropped}")
        return tuple(kept[:MAX_OBSERVATION_REFS])

    def request(
        self, request: dict[str, Any], reason: str, based_on: list[str]
    ) -> dict[str, Any]:
        """Send one tool request through the gateway and record the step."""
        if self.packet is not None:
            return {"ok": False, "error": "the run is finished"}
        if len(self.steps) >= MAX_TURNS - 1:
            return {"ok": False, "error": "budget exhausted: call finish now"}
        reply = self.gateway.call(request)
        self.requests.append(
            {"request": request, "reason": reason, "based_on": based_on}
        )
        step = TraceStep(
            turn=len(self.steps) + 1,
            summary=(reason.strip() or "no reason given")[:280],
            observation_refs=self._refs(based_on),
            call=reply.call,
            result=reply.result if reply.call is not None else None,
        )
        if reply.call is None:
            self.notes.append(
                f"turn {step.turn}: refused request: {reply.result.error}"
            )
        self.steps.append(step)
        seen = reply.result.model_dump(mode="json", exclude={"result_hash"})
        seen["fields"].pop("snapshot_id", None)  # identity, not evidence
        return seen

    def finish(
        self,
        outcome: str,
        claims: list[ClaimInput | dict[str, Any]],
        not_established: list[str],
        missing: list[str],
        reason: str,
        based_on: list[str],
    ) -> dict[str, Any]:
        """Assemble the packet; accept it only if it passes every check."""
        try:
            state = TerminalState(outcome)
            parsed = [ClaimInput.model_validate(claim) for claim in claims]
            clarification = None
            if state is TerminalState.CLARIFY:
                scopes = frozenset(MissingScope(name) for name in missing)
                options = (
                    {MissingScope.METRIC: tuple(metric.value for metric in Metric)}
                    if MissingScope.METRIC in scopes
                    else {}
                )
                clarification = Clarification(missing=scopes, options=options)
            final = TraceStep(
                turn=len(self.steps) + 1,
                summary=(reason.strip() or "finish")[:280],
                observation_refs=self._refs(based_on),
            )
            trace = (*self.steps, final)
            packet = Packet(
                case_id=self.case.case_id,
                runner="agent",
                snapshot_id=self.case.snapshot_id,
                terminal_state=state,
                claims=tuple(
                    Claim(
                        label=ClaimLabel(claim.label),
                        template=claim.text,
                        row_refs=tuple(claim.rows),
                    )
                    for claim in parsed
                ),
                clarification=clarification,
                not_established=tuple(not_established),
                trace=trace,
                cost=Cost(
                    turns=len(trace),
                    calls=sum(s.call is not None for s in trace),
                    usd=0,
                    seconds=0,
                ),
            )
        except (ValueError, ValidationError) as exc:
            self.notes.append(f"finish refused: {exc}")
            return {"ok": False, "error": str(exc)[:800]}
        stray = packet_stray_digits(
            packet, known_identifiers(self.case, self.snapshot, packet)
        )
        if stray:
            self.notes.append(f"finish refused: stray digits {stray}")
            return {
                "ok": False,
                "error": f"typed numbers are not allowed, use pointer tokens: {stray}",
            }
        self.packet = packet
        return {"ok": True}

    def incomplete(self, why: str) -> Packet:
        """The packet when the model never finished validly: INCOMPLETE, no claims."""
        steps = self.steps[: MAX_TURNS - 1]
        trace = (*steps, TraceStep(turn=len(steps) + 1, summary=why))
        return Packet(
            case_id=self.case.case_id,
            runner="agent",
            snapshot_id=self.case.snapshot_id,
            terminal_state=TerminalState.INCOMPLETE,
            trace=trace,
            cost=Cost(
                turns=len(trace),
                calls=sum(s.call is not None for s in trace),
                usd=0,
                seconds=0,
            ),
        )

## The tools the model sees

Plain functions with typed arguments and a docstring, which ADK turns into tool
declarations. Day ranges are flat start and end days; each function builds the
gateway request and returns the gateway's reply.

In [ ]:
#| export
def model_tools(run: AgentRun) -> list[Any]:
    """The five tools and `finish`, bound to one run."""

    def resolve_context(reason: str, based_on: list[str]) -> dict[str, Any]:
        """Days, series, model versions, issue days and metrics in this snapshot."""
        return run.request({"tool": "resolve_context"}, reason, based_on)

    def compare_runs(
        series: list[str],
        metric: str,
        period_start: int,
        period_end: int,
        window_start: int,
        window_end: int,
        reason: str,
        based_on: list[str],
        model_version: str = "",
    ) -> dict[str, Any]:
        """Per series: MAE and bias in the review period and in the earlier window, their
        change, actual rows found against days expected, and the model version and
        horizons used.
        metric is mae or signed_bias. model_version is optional (default: the model under
        review, not the seasonal_naive_v1 benchmark)."""
        request: dict[str, Any] = {
            "tool": "compare_runs",
            "series": series,
            "metric": metric,
            "period": _range(period_start, period_end),
            "window": _range(window_start, window_end),
        }
        if model_version:
            request["model_version"] = model_version
        return run.request(request, reason, based_on)

    def inspect_series(
        series_id: str, start_day: int, end_day: int, reason: str, based_on: list[str]
    ) -> dict[str, Any]:
        """One series day by day: actual, forecast, horizon, model; with averages and
        recent bias."""
        request = {
            "tool": "inspect_series",
            "series_id": series_id,
            "days": _range(start_day, end_day),
        }
        return run.request(request, reason, based_on)

    def check_quality(
        series: list[str],
        start_day: int,
        end_day: int,
        reason: str,
        based_on: list[str],
    ) -> dict[str, Any]:
        """Per series: missing days, duplicate rows, coverage, how many model version
        labels the forecasts carry, and the horizons they were made at; plus the IDs of
        the affected rows."""
        request = {
            "tool": "check_quality",
            "series": series,
            "days": _range(start_day, end_day),
        }
        return run.request(request, reason, based_on)

    def evidence_rows(
        row_ids: list[str], reason: str, based_on: list[str]
    ) -> dict[str, Any]:
        """The exact rows with these IDs (at most fifty)."""
        return run.request(
            {"tool": "evidence_rows", "row_ids": row_ids}, reason, based_on
        )

    def finish(
        outcome: str,
        claims: list[ClaimInput],
        not_established: list[str],
        missing: list[str],
        reason: str,
        based_on: list[str],
        tool_context: ToolContext,
    ) -> dict[str, Any]:
        """End the run. outcome: ANSWER, CLARIFY, STOP_UNTRUSTWORTHY or INCOMPLETE.
        claims: FINDING (cite row IDs in rows) or INFERENCE, numbers only as pointer tokens.
        missing: for CLARIFY, any of metric, period, window, series; otherwise empty.
        If the reply is not ok, fix what it says and call finish again."""
        if len(run.steps) >= MAX_TURNS:
            return {"ok": False, "error": "no turns left"}
        reply = run.finish(
            outcome, list(claims), not_established, missing, reason, based_on
        )
        if reply["ok"]:
            tool_context.actions.skip_summarization = True
        return reply

    return [
        resolve_context,
        compare_runs,
        inspect_series,
        check_quality,
        evidence_rows,
        finish,
    ]

## Running a case

One fresh ADK session per run. The model gets at most `MAX_TURNS + 2` calls (a refused
`finish` can be retried); if no valid packet results, the run ends INCOMPLETE. Tokens are
summed from ADK's events and priced with `USD_PER_TOKEN`.

The model sees only the review day and the request, not the case ID, so a matched pair
gets identical input.

In [ ]:
#| export
async def run_agent_async(
    case: Case,
    snapshot: Snapshot,
    setting: RunSetting | None = None,
    model: str = MODEL,
) -> dict[str, Any]:
    """Run the agent on one case and return its run record (packet included)."""
    run = AgentRun(case, snapshot, setting)
    agent = LlmAgent(
        name="forecast_reviewer",
        model=LiteLlm(
            model=model, api_key=openrouter_api_key(), temperature=TEMPERATURE
        ),
        instruction=INSTRUCTION,
        tools=model_tools(run),
    )
    runner = InMemoryRunner(agent=agent, app_name="forecast_review")
    session = await runner.session_service.create_session(
        app_name="forecast_review", user_id="analyst"
    )
    message = types.Content(
        role="user",
        parts=[
            types.Part(
                text=f"Review day: {case.review_day}.\nRequest: {case.request_text}"
            )
        ],
    )
    tokens = {"prompt": 0, "completion": 0}
    started = time.monotonic()
    error = None
    try:
        async for event in runner.run_async(
            user_id="analyst",
            session_id=session.id,
            new_message=message,
            run_config=RunConfig(max_llm_calls=MAX_TURNS + 2),
        ):
            if event.usage_metadata is not None:
                tokens["prompt"] += event.usage_metadata.prompt_token_count or 0
                tokens["completion"] += event.usage_metadata.candidates_token_count or 0
    except Exception as exc:  # noqa: BLE001  any failure must still leave a record
        error = f"{type(exc).__name__}: {exc}"[:500]
    seconds = round(time.monotonic() - started, 1)
    usd = round(sum(tokens[kind] * USD_PER_TOKEN[kind] for kind in tokens), 6)
    packet = run.packet or run.incomplete("no valid finish within the budget")
    packet = packet.model_copy(
        update={"cost": packet.cost.model_copy(update={"usd": usd, "seconds": seconds})}
    )
    return {
        "case_id": case.case_id,
        "snapshot_id": case.snapshot_id,
        "model": model,
        "temperature": TEMPERATURE,
        "prompt_sha256": PROMPT_SHA256,
        "started_at": datetime.now(UTC).isoformat(timespec="seconds"),
        "tokens": tokens,
        "error": error,
        "notes": run.notes,
        "requests": run.requests,
        "packet": packet.model_dump(mode="json"),
    }


def run_agent(
    case: Case, snapshot: Snapshot, setting: RunSetting | None = None
) -> dict[str, Any]:
    """`run_agent_async` for plain scripts (in a notebook, `await` the async version)."""
    return asyncio.run(run_agent_async(case, snapshot, setting))

## Saving and replaying

Records are saved as `runs/<split>/<case_id>/<run_id>.json`. `replay` needs no model:
it re-issues each recorded request through a fresh gateway (same case, snapshot and run
setting), checks every reply against the packet's trace (call name and fingerprint),
and returns the validated packet.

In [ ]:
#| export
def save_run(record: Mapping[str, Any], root: Path, split: str = "dev") -> Path:
    """Write a run record and return its path."""
    folder = root / split / str(record["case_id"])
    folder.mkdir(parents=True, exist_ok=True)
    stamp = str(record["started_at"]).replace(":", "").replace("+0000", "Z")
    path = folder / f"{stamp}_{len(list(folder.glob('*.json'))) + 1}.json"
    path.write_text(
        json.dumps(record, indent=1, ensure_ascii=False) + "\n", encoding="utf-8"
    )
    return path


def replay(
    record: Mapping[str, Any],
    case: Case,
    snapshot: Snapshot,
    setting: RunSetting | None = None,
) -> Packet:
    """Re-run a saved record's requests offline; raise if any reply differs."""
    packet = Packet.model_validate(record["packet"])
    results = iter(step.result for step in packet.trace if step.result is not None)
    gateway = Gateway(case, snapshot, TOOLS, setting)
    for entry in record["requests"]:
        reply = gateway.call(entry["request"])
        if reply.call is None:
            continue
        saved = next(results)
        if (reply.result.call_id, reply.result.result_hash) != (
            saved.call_id,
            saved.result_hash,
        ):
            raise ValueError(f"replay differs at {saved.call_id}")
    if next(results, None) is not None:
        raise ValueError("the record has fewer requests than the trace")
    return packet

## Examples

Without a model: an `AgentRun` driven by hand, the way the model would drive it. The
finish below types a number, so it is refused; the corrected one with a pointer token is
accepted, and the packet replays.

In [ ]:
from agentic_forecast_review.cases import build_family

family = build_family()
case = family.cases[1]
snapshot = family.snapshots[case.snapshot_id]
run = AgentRun(case, snapshot)
reply = run.request(
    {"tool": "check_quality", "series": ["CA_2_FOODS"], "days": _range(169, 196)},
    "check the data before judging the model",
    [],
)
print(reply["call_id"], reply["fields"])
rows = list(reply["row_ids"])
typed = run.finish(
    "STOP_UNTRUSTWORTHY",
    [{"label": "FINDING", "text": "4 actual days are missing.", "rows": rows}],
    ["whether the model got worse"],
    [],
    "actual days are missing",
    ["dq1"],
)
print(typed)
assert not typed["ok"]
accepted = run.finish(
    "STOP_UNTRUSTWORTHY",
    [
        {
            "label": "FINDING",
            "text": "{ev:dq1.ca_2_foods__missing_days} actual days are missing.",
            "rows": rows,
        }
    ],
    ["whether the model got worse"],
    [],
    "actual days are missing",
    ["dq1"],
)
assert accepted["ok"] and run.packet is not None
print(run.packet.render())
record = {"requests": run.requests, "packet": run.packet.model_dump(mode="json")}
assert replay(record, case, snapshot) == run.packet

A live run (needs the key; skipped by CI). The packet, its trace and its cost:

In [ ]:
#| notest
live = await run_agent_async(case, snapshot)
packet = Packet.model_validate(live["packet"])
print(packet.terminal_state, live["tokens"], packet.cost.usd)
for step in packet.trace:
    print(
        step.turn,
        step.call.args.tool if step.call else "finish",
        step.observation_refs,
        step.summary,
    )
for line in packet.render():
    print("-", line)